In [0]:
%run ../delta_function

In [0]:
import os
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import TimestampType
import pyspark.sql.utils;
from pyspark.sql.types import StructType, StringType;
from pyspark.sql.functions import concat, lit, col, upper, max, when, row_number, to_date
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from datetime import datetime, timedelta
from pyspark.sql.functions import regexp_replace
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_date, datediff, when
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [0]:
try:
    verbose_mode = dbutils.widgets.get("verbose_mode");
except:
    verbose_mode = 'debug'
try:
    current_division = dbutils.widgets.get("division");
except:
    current_division = 'mal'
try:
    current_environment = dbutils.widgets.get("environment");
except:
    current_environment = 'dev'
try:
    execution_mode = dbutils.widgets.get("execution_mode");
except:
    execution_mode = 'update'
try:
    current_project = dbutils.widgets.get("project");
except:
    current_project = 'maite_bi'
try:
    current_production_line = dbutils.widgets.get("production_line");
except:
    current_production_line = 'recommendations'

current_catalog = current_division + '_' + current_project + '_' + current_environment;

current_schema = current_production_line;

current_location = 'abfss://' + current_project + '@adlsdpcom'+ current_environment +f'data.dfs.core.windows.net/' + current_production_line + '/'

if verbose_mode == 'debug':
    display("Debug Mode")
    display(f"current_division : {current_division}")
    display(f"current_environment : {current_environment}")
    display(f"current_project : {current_project}")
    display(f"current_production_line : {current_production_line}")
    display(f"current_catalog : {current_catalog}")
    display(f"current_schema : {current_schema}")
    display(f"current_location : {current_location}")

Source(s)

In [0]:
if current_environment =='preprd':
    source = f"""ext_mal_psql_maite_vision_board_test.public"""    
else:
    source = f"""ext_mal_psql_maite_vision_board_{current_environment}.public"""

batches

In [0]:
batches = f"""
SELECT id_batch, batch_number, b.production_line AS id_plant, ppl.name AS plant, requirement_specifications AS id_cdc, b.variety AS id_variety, b.planned_date, b.mes_number, pbst.label AS batch_status, harvest
FROM {source}.batches b
JOIN {source}.plants_production_lines ppl ON ppl.id_plant_production_line = b.production_line
JOIN {source}.parameters_batch_status_translations pbst ON pbst.status = b.status AND pbst.language = 1
WHERE b.deleted = false
"""

df_batches = spark.sql(batches)
df_batches.createOrReplaceTempView("batches")

In [0]:
reco_opti = f"""
SELECT batch, COUNT(batch) AS nb_reco_opti
FROM {source}.batches_production_planning bpp
WHERE bpp.deleted = false
GROUP BY batch
"""

df_reco_opti = spark.sql(reco_opti)
df_reco_opti.createOrReplaceTempView("reco_opti")

In [0]:
reco = f"""
SELECT r.id_recommendation, r.batch, r.calculation_date, r.status, r.acceptance_user, r.acceptance_comments, r.target_localization, r.calculation_localization, r.acceptance_status, b.plant, plt.label AS localization
FROM {source}.recommendations r
JOIN batches b ON b.id_batch = r.batch
JOIN {source}.parameters_localizations_translations plt ON plt.id_parameter_localization = r.target_localization
WHERE plt.language = 1 AND r.deleted = false
"""

df_reco = spark.sql(reco)

# Supprimer les lignes par condition
df_reco = df_reco.filter(~((F.col("plant") == "ROUEN1") & (F.col("localization") == "Germination J5")))
#df_reco = df_reco.filter(~((F.col("plant") == "NOGENT2") & (F.col("localization") == "Trempe J2")))        Retiré le 18/09/2024
#df_reco = df_reco.filter(~((F.col("plant") == "POLISY1") & (F.col("localization") == "Trempe J2")))        Retiré le 18/09/2024

batches_reco

In [0]:
reco_final = f"""
SELECT r.batch, b.production_line, r.target_localization, r.acceptance_status
FROM {source}.recommendations r
JOIN {source}.batches b ON b.id_batch = r.batch AND b.deleted = false
WHERE r.deleted = false
"""

df_reco_final = spark.sql(reco_final)

In [0]:
# Supprimer les recommandations Germination J5 sur ROUEN1
df_reco_final = df_reco_final.filter(~((F.col("production_line") == 1) & (F.col("target_localization") == 14)))

In [0]:
# Créer une fenêtre partitionnée par batch_number et target_localization
windowSpec = Window.partitionBy("batch", "target_localization").orderBy(
    when(col("acceptance_status") == 1, 1)
    .when(col("acceptance_status") == -1, 2)
    .otherwise(3)
)

# Ajouter une colonne de rang en fonction de la priorité sur acceptance_status
df_ranked = df_reco_final.withColumn("rank", row_number().over(windowSpec))

# Garder seulement la première ligne par groupe
df_reco_final = df_ranked.filter(col("rank") == 1).drop("rank")

# GroupBy sur 'batch' et compter les occurrences, puis renommer la colonne 'count' en 'total'
df_reco_final = df_reco_final.groupBy("batch").count().withColumnRenamed("count", "total")

In [0]:
# Supprimer les espaces dans la colonne "acceptance_comments"
df_reco = df_reco.withColumn("acceptance_comments_clean",F.regexp_replace(F.col("acceptance_comments"), "\\s+", ""))

# Ajouter la colonne "applied_status" en utilisant la colonne nettoyée
df_reco = df_reco.withColumn(
    "applied_status",
    F.when(
        (F.col("acceptance_status") == 1) & 
        (F.col("acceptance_comments_clean").rlike("(?i)appli|apli|aplli")) &
        (~F.col("acceptance_comments_clean").rlike("(?i)non|no|pas")),
        "acceptee_et_appliquee"
    ).when(
        F.col("acceptance_status") == 1,
        "acceptee_mais_non_appliquee"
    ).when(
        F.col("acceptance_status") == -1,
        "refusee_ia"
    ).when(
        F.col("acceptance_status") == -2,
        "refusee_usine"
    ).when(
        F.col("acceptance_status") == 0,
        "en_attente"
    ).otherwise(F.lit(None)))

df_reco_save = df_reco.drop("acceptance_comments_clean")

In [0]:
# Sélectionner uniquement les colonnes 'batch', 'target_localization' et 'acceptance_status'
df_reco_batch = df_reco_save.select("batch", "target_localization", "acceptance_status", "applied_status")

# Ajouter la colonne 'status' en fonction de la valeur de 'acceptance_status'
df_reco_batch = df_reco_batch.withColumn(
    "status",
    F.when(F.col("acceptance_status") == -1, "refusee_ia")
     .when(F.col("acceptance_status") == -2, "refusee_usine")
     .when(F.col("acceptance_status") == 0, "en_attente")
     .when(F.col("acceptance_status") == 1, "acceptee"))

# Pivot sur la colonne 'applied_status'
df_reco_batch_applied = df_reco_batch.withColumn("nb_reco", F.lit(1))
df_reco_batch_applied = df_reco_batch_applied.filter(
    (F.col("applied_status") == "acceptee_et_appliquee") |
    (F.col("applied_status") == "acceptee_mais_non_appliquee"))

df_reco_batch_applied = df_reco_batch_applied.groupBy("batch") \
                                            .pivot("applied_status") \
                                            .agg(F.sum("nb_reco"))

# Pivot sur la colonne 'status'
df_reco_batch = df_reco_batch.withColumn("nb_reco", F.lit(1))
df_reco_batch = df_reco_batch.groupBy("batch") \
                              .pivot("status") \
                              .agg(F.sum("nb_reco"))


# À supprimer dès qu'il y aura des valeurs -2
# Vérifie si la colonne existe
if 'refusee_usine' not in df_reco_batch.columns:
    df_reco_batch = df_reco_batch.withColumn('refusee_usine', lit(0))


# Jointure
df_reco_batch = df_reco_batch.join(df_reco_batch_applied.select("batch", "acceptee_et_appliquee", "acceptee_mais_non_appliquee"),
                                   df_reco_batch.batch == df_reco_batch_applied.batch,
                                   how="left") \
                             .drop(df_reco_batch_applied.batch)

# Remplacer les null par des 0
df_reco_batch = df_reco_batch.fillna(0, subset=["acceptee", "en_attente", "refusee_ia", "refusee_usine", "acceptee_et_appliquee", "acceptee_mais_non_appliquee"])

# Ajouter la colonne 'evaluee' avec la somme des colonnes 'acceptee' et 'refusee'
df_reco_batch = df_reco_batch.withColumn("evaluee",F.col("acceptee") + F.col("refusee_ia") + F.col("refusee_usine"))

# Ajouter la colonne 'total_all' avec la somme des colonnes 'acceptee', 'en_attente', et 'refusee'
df_reco_batch = df_reco_batch.withColumn("total_all",F.col("acceptee") + F.col("en_attente") + F.col("refusee_ia") + F.col("refusee_usine"))

# Jointure
df_batches_reco = df_batches.join(df_reco_batch.select("batch", "acceptee", "en_attente", "refusee_ia", "refusee_usine", "acceptee_et_appliquee", "acceptee_mais_non_appliquee", "evaluee", "total_all"),
                                   df_batches.id_batch == df_reco_batch.batch,
                                   how="left").drop(df_reco_batch.batch)

# Jointure pour la colonne 'total'
df_batches_reco = df_batches_reco.join(df_reco_final, df_batches_reco["id_batch"] == df_reco_final["batch"], how="left").drop("batch")

# Jointure pour la colonne 'nb_reco_opti'
df_batches_reco = df_batches_reco.join(df_reco_opti, df_batches_reco["id_batch"] == df_reco_opti["batch"], how="left").drop("batch")

# Remplacer les null par des 0
df_batches_reco = df_batches_reco.fillna(0, subset=["acceptee", "en_attente", "refusee_ia", "refusee_usine", "acceptee_et_appliquee", "acceptee_mais_non_appliquee", "evaluee", "total_all", "total", "nb_reco_opti"])

In [0]:
# Sélectionne les colonnes suivantes
df_batches_reco = df_batches_reco.select("id_batch", "planned_date", "nb_reco_opti", "acceptee", "en_attente", "refusee_ia", "refusee_usine", "acceptee_et_appliquee", "acceptee_mais_non_appliquee", "evaluee", "total_all", "total")

# Trie croissant sur les colonnes suivantes
df_batches_reco = df_batches_reco.orderBy("planned_date")

Import fact_batches_reco

In [0]:
current_process="fact_batches_reco"

In [0]:
target_fact_batches_reco = current_catalog +"."+current_schema+"."+current_process
print(target_fact_batches_reco)

In [0]:
all_columns =  df_batches_reco.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_batch']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_batches_reco, 
    target_fact_batches_reco, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

recommendations

In [0]:
reco_max = f"""
SELECT id_recommendation, batch, status, target_localization, calculation_date
FROM {source}.recommendations
WHERE deleted = false
"""

df_reco_max = spark.sql(reco_max)

# Grouper par 'batch_number' et 'target_localization', pour compter les occurrences
df_counts = df_reco_max.groupBy("batch", "target_localization").count()

# Joindre le DataFrame original avec les comptes pour ajouter une nouvelle colonne
df_reco_max = df_reco_max.join(df_counts, on=["batch", "target_localization"], how="left")

# Renommer la colonne 'count'
df_reco_max = df_reco_max.withColumnRenamed("count", "batch_target_count")

# Modifier la colonne 'status' en fonction de la valeur de 'batch_target_count'
df_reco_max = df_reco_max.withColumn(
    "status",
    F.when(F.col("batch_target_count") == 1, F.lit(1)).otherwise(F.col("status"))
)

# Filtrer les lignes où 'status' est égal à 1
df_reco_max = df_reco_max.filter(F.col("status") == 1)

# Grouper par 'batch_number' et 'target_localization', et prendre la date max de 'calculation_date'
df_reco_max = df_reco_max.groupBy("batch", "target_localization") \
                        .agg(F.max("calculation_date").alias("max_calculation_date"))

# Ajouter une colonne "Reco - Date Max" avec la valeur 1
df_reco_max = df_reco_max.withColumn("Reco - Date Max", F.lit(1))

In [0]:
# Joindre df_reco avec df_reco_max en utilisant les colonnes spécifiées
df_reco = df_reco.join(df_reco_max,
                        on=[df_reco.batch == df_reco_max.batch,
                            df_reco.target_localization == df_reco_max.target_localization,
                            df_reco.calculation_date == df_reco_max.max_calculation_date],
                        how="left")

df_reco = df_reco.drop(df_reco_max.batch) \
                   .drop(df_reco_max.target_localization) \
                   .drop(df_reco_max.max_calculation_date)

# Filtrer les lignes où "Reco - Date Max" est égal à 1
df_reco = df_reco.filter(F.col("Reco - Date Max") == 1)

# Ajouter la colonne 'status' en fonction de la valeur de 'acceptance_status'
df_reco = df_reco.withColumn(
    "acceptance_status",
    F.when(F.col("applied_status") == "refusee_ia", -1)
     .when(F.col("applied_status") == "refusee_usine", -2)
     .when(F.col("applied_status") == "en_attente", 0)
     .when(F.col("applied_status") == "acceptee_mais_non_appliquee", 1)
     .when(F.col("applied_status") == "acceptee_et_appliquee", 2))

# Ajout de la colonne avec la date du jour
df_reco = df_reco.withColumn("today", current_date())

# Calcul du nombre de jours entre date_du_jour et calculation_date
df_reco = df_reco.withColumn("days_before", datediff(col("today"), col("calculation_date")))

# Créer une vue temporaire pour le DataFrame
df_reco.createOrReplaceTempView("reco")

In [0]:
# Ajout des notes de production
reco_notes = f"""
SELECT bn.batch, bn.note AS note_1, pa.login AS user_note_1
FROM {source}.batches_notes bn
JOIN {source}.profiles_accounts pa ON pa.id_profile_account = bn.profile_account AND pa.deleted = false
WHERE bn.deleted = false
"""

df_reco_notes = spark.sql(reco_notes)

In [0]:
# Ajout d'un numéro de ligne pour différencier les doublons
window_spec = Window.partitionBy("batch").orderBy("batch")
df_with_rownum = df_reco_notes.withColumn("row_num", row_number().over(window_spec))

# Séparer les doublons en note_2 et user_note_2
df_notes_2 = df_with_rownum.filter(col("row_num") == 2).select(
    col("batch").alias("batch_2"),
    col("note_1").alias("note_2"),
    col("user_note_1").alias("user_note_2")
)

# Séparer les doublons en note_3 et user_note_3
df_notes_3 = df_with_rownum.filter(col("row_num") == 3).select(
    col("batch").alias("batch_3"),
    col("note_1").alias("note_3"),
    col("user_note_1").alias("user_note_3")
)

# Garder uniquement la première occurrence
df_notes_1 = df_with_rownum.filter(col("row_num") == 1).drop("row_num")

# Fusionner les deux dataframes
df_reco_notes = df_notes_1.join(df_notes_2, df_notes_1.batch == df_notes_2.batch_2, "left") \
    .drop("batch_2")
df_reco_notes = df_reco_notes.join(df_notes_3, df_reco_notes.batch == df_notes_3.batch_3, "left") \
    .drop("batch_3")

# Créer une vue temporaire pour le DataFrame
df_reco_notes.createOrReplaceTempView("reco_notes")

In [0]:
reco = f"""
SELECT reco.*, reco_notes.note_1, reco_notes.user_note_1, reco_notes.note_2, reco_notes.user_note_2, reco_notes.note_3, reco_notes.user_note_3
FROM reco
LEFT JOIN reco_notes ON reco_notes.batch = reco.batch
"""

df_reco = spark.sql(reco)

In [0]:
# Supprimer les colonnes suivantes
df_reco = df_reco.select("id_recommendation", "batch", "calculation_date", "acceptance_user", "acceptance_comments", "target_localization", "acceptance_status", "days_before", "note_1", "user_note_1", "note_2", "user_note_2", "note_3", "user_note_3")

# Remplir les cellules vides des colonnes avec "-"
df_reco = df_reco.fillna("-", subset=["note_1", "user_note_1", "note_2", "user_note_2", "note_3", "user_note_3"])

Import fact_recommendations

In [0]:
current_process="fact_recommendations"

In [0]:
target_fact_recommendations = current_catalog +"."+current_schema+"."+current_process
print(target_fact_recommendations)

In [0]:
all_columns =  df_reco.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_recommendation']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_reco, 
    target_fact_recommendations, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

batches_localization_reco

In [0]:
# Dataframe pour les batches et leur localisation
batches_localization = f"""
SELECT b.id_batch, b.id_plant, b.plant, id_parameter_localization, plt.label AS localization
FROM batches b
CROSS JOIN {source}.parameters_localizations_translations plt
WHERE language = 1 AND id_parameter_localization >= 8 AND id_parameter_localization <= 20 AND id_parameter_localization NOT IN (15, 18, 19)
"""

df_batches_localization = spark.sql(batches_localization)

In [0]:
reco_localization = f"""
SELECT r.batch, r.id_recommendation, plt.label AS localization, r.acceptance_status, r.calculation_date
FROM {source}.recommendations r
JOIN {source}.parameters_localizations_translations plt ON plt.id_parameter_localization = r.target_localization
WHERE r.deleted = false AND plt.language = 1
"""

df_reco_localization = spark.sql(reco_localization)

# Ajouter la colonne 'status' en fonction de la valeur de 'acceptance_status'
df_reco_localization = df_reco_localization.withColumn(
    "status",
    F.when(F.col("acceptance_status") == -1, "refusee_ia")
     .when(F.col("acceptance_status") == -2, "refusee_usine")
     .when(F.col("acceptance_status") == 0, "en_attente")
     .when(F.col("acceptance_status") == 1, "acceptee"))

# Convertir les colonnes en format timestamp
df_reco_localization = df_reco_localization.withColumn("calculation_date", col("calculation_date").cast("timestamp"))
df_reco_localization = df_reco_localization.withColumn("calculation_date", F.to_date("calculation_date"))

# Pivot sur la colonne 'status'
df_reco_localization = df_reco_localization.withColumn("nb_reco", F.lit(1))
df_reco_localization = df_reco_localization.groupBy("batch", "id_recommendation", "localization", "calculation_date") \
                              .pivot("status") \
                              .agg(F.sum("nb_reco"))


# À supprimer dès qu'il y aura des valeurs -2
# Vérifie si la colonne existe
if 'refusee_usine' not in df_reco_localization.columns:
    df_reco_localization = df_reco_localization.withColumn('refusee_usine', lit(0))


# Remplacer les null par des 0
df_reco_localization = df_reco_localization.fillna(0, subset=["acceptee", "en_attente", "refusee_ia", "refusee_usine"])

# Ajouter la colonne 'evaluee' avec la somme des colonnes 'acceptee' et 'refusee'
df_reco_localization = df_reco_localization.withColumn("evaluee",F.col("acceptee") + F.col("refusee_ia") + F.col("refusee_usine"))

# Ajouter la colonne 'total' avec la somme des colonnes 'acceptee', 'en_attente', et 'refusee'
df_reco_localization = df_reco_localization.withColumn("total",F.col("acceptee") + F.col("en_attente") + F.col("refusee_ia") + F.col("refusee_usine"))

In [0]:
# Sélectionner uniquement les colonnes 'batch', 'target_localization' et 'acceptance_status'
df_reco_applied = df_reco_save.select("batch", "id_recommendation", "target_localization", "applied_status")

# Pivot sur la colonne 'applied_status'
df_reco_applied = df_reco_applied.withColumn("nb_reco", F.lit(1))
df_reco_applied = df_reco_applied.filter(
    (F.col("applied_status") == "acceptee_et_appliquee") |
    (F.col("applied_status") == "acceptee_mais_non_appliquee")
)
df_reco_applied = df_reco_applied.groupBy("batch", "id_recommendation", "target_localization") \
                                            .pivot("applied_status") \
                                            .agg(F.sum("nb_reco"))

In [0]:
# Jointure pour les colonnes "acceptee", "en_attente", "refusee", "evaluee" et "total"
df_batches_localization = df_batches_localization.join(df_reco_localization,
                        on=[df_batches_localization.id_batch == df_reco_localization.batch,
                            df_batches_localization.localization == df_reco_localization.localization],
                        how="left")

df_batches_localization = df_batches_localization.drop(df_reco_localization.batch) \
                    .drop(df_reco_localization.batch) \
                    .drop(df_reco_localization.localization)

# Jointure pour les colonnes "acceptee_et_appliquee" et "acceptee_mais_non_appliquee"
df_batches_localization = df_batches_localization.join(df_reco_applied,
                        on=[df_batches_localization.id_batch == df_reco_applied.batch,
                            df_batches_localization.id_recommendation == df_reco_applied.id_recommendation,
                            df_batches_localization.id_parameter_localization == df_reco_applied.target_localization],
                        how="left")

df_batches_localization = df_batches_localization.drop(df_reco_applied.batch) \
                    .drop(df_reco_applied.id_recommendation) \
                    .drop(df_reco_applied.target_localization)

# Remplacer les null par des 0
df_batches_localization = df_batches_localization.fillna(0, subset=["acceptee", "en_attente", "refusee_ia", "refusee_usine", "evaluee", "total", "acceptee_et_appliquee", "acceptee_mais_non_appliquee"])

# Garde les lignes avec Pré-germination pour Prouvy et supprime les autres pour cette localisation
df_batches_localization = df_batches_localization.filter(
    (col("localization") != "Pré-germination") | 
    ((col("localization") == "Pré-germination") & (col("plant") == "PROUVY1")))

In [0]:
df_batches_localization = df_batches_localization.withColumn('calculation_date',when(df_batches_localization['calculation_date'].isNull(), lit('1999-01-01'))
                                                                .otherwise(df_batches_localization['calculation_date']))


df_batches_localization = df_batches_localization.withColumn('calculation_date',to_date('calculation_date', 'yyyy-MM-dd'))

In [0]:
# Supprimer les colonnes suivantes
df_batches_localization = df_batches_localization.drop("id_plant", "plant", "localization")

# Renommer les colonnes suivantes
df_batches_localization = df_batches_localization.withColumnRenamed("id_parameter_localization", "id_localization")

Import fact_batches_localization_reco

In [0]:
current_process="fact_batches_localization_reco"

In [0]:
target_fact_batches_localization = current_catalog +"."+current_schema+"."+current_process
print(target_fact_batches_localization)

In [0]:
all_columns =  df_batches_localization.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_batch'
    ,'id_localization'
    ,'id_recommendation'
    ,'calculation_date']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_batches_localization, 
    target_fact_batches_localization, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

batches_calculation_date_reco

In [0]:
reco_calculation_date = f"""
SELECT r.batch, r.calculation_date, r.acceptance_status
FROM {source}.recommendations r
WHERE r.deleted = false
"""

df_reco_calculation_date = spark.sql(reco_calculation_date)

# Ajouter la colonne 'status' en fonction de la valeur de 'acceptance_status'
df_reco_calculation_date = df_reco_calculation_date.withColumn(
    "status",
    F.when(F.col("acceptance_status") == -1, "refusee_ia")
     .when(F.col("acceptance_status") == -2, "refusee_usine")
     .when(F.col("acceptance_status") == 0, "en_attente")
     .when(F.col("acceptance_status") == 1, "acceptee"))

# Convertir les colonnes en format timestamp
df_reco_calculation_date = df_reco_calculation_date.withColumn("calculation_date", col("calculation_date").cast("timestamp"))
df_reco_calculation_date = df_reco_calculation_date.withColumn("calculation_date", F.to_date("calculation_date"))

# Pivot sur la colonne 'status'
df_reco_calculation_date = df_reco_calculation_date.withColumn("nb_reco", F.lit(1))
df_reco_calculation_date = df_reco_calculation_date.groupBy("batch", "calculation_date") \
                                                    .pivot("status") \
                                                    .agg(F.sum("nb_reco"))


# À supprimer dès qu'il y aura des valeurs -2
# Vérifie si la colonne existe
if 'refusee_usine' not in df_reco_calculation_date.columns:
    df_reco_calculation_date = df_reco_calculation_date.withColumn('refusee_usine', lit(0))


# Remplacer les null par des 0
df_reco_calculation_date = df_reco_calculation_date.fillna(0, subset=["acceptee", "en_attente", "refusee_ia", "refusee_usine"])

# Ajouter la colonne 'evaluee' avec la somme des colonnes 'acceptee' et 'refusee'
df_reco_calculation_date = df_reco_calculation_date.withColumn("evaluee",F.col("acceptee") + F.col("refusee_ia") + F.col("refusee_usine"))

# Ajouter la colonne 'total' avec la somme des colonnes 'acceptee', 'en_attente', et 'refusee'
df_reco_calculation_date = df_reco_calculation_date.withColumn("total",F.col("acceptee") + F.col("en_attente") + F.col("refusee_ia") + F.col("refusee_usine"))

In [0]:
df_reco_applied_date = df_reco_save.select("batch", "calculation_date", "applied_status")

# Convertir les colonnes en format timestamp
df_reco_applied_date = df_reco_applied_date.withColumn("calculation_date", col("calculation_date").cast("timestamp"))
df_reco_applied_date = df_reco_applied_date.withColumn("calculation_date", F.to_date("calculation_date"))

# Pivot sur la colonne 'applied_status'
df_reco_applied_date = df_reco_applied_date.withColumn("nb_reco", F.lit(1))
df_reco_applied_date = df_reco_applied_date.filter(
    (F.col("applied_status") == "acceptee_et_appliquee") |
    (F.col("applied_status") == "acceptee_mais_non_appliquee"))

df_reco_applied_date = df_reco_applied_date.groupBy("batch", "calculation_date") \
                                            .pivot("applied_status") \
                                            .agg(F.sum("nb_reco"))

# Jointure pour les colonnes "acceptee_et_appliquee" et "acceptee_mais_non_appliquee"
df_batches_calculation_date_reco = df_reco_calculation_date.join(df_reco_applied_date,
                        on=[df_reco_calculation_date.batch == df_reco_applied_date.batch,
                            df_reco_calculation_date.calculation_date == df_reco_applied_date.calculation_date],
                        how="left")

df_batches_calculation_date_reco = df_batches_calculation_date_reco.drop(df_reco_applied_date.batch) \
                    .drop(df_reco_applied_date.calculation_date)

# Remplacer les null par des 0
df_batches_calculation_date_reco = df_batches_calculation_date_reco.fillna(0, subset=["acceptee_et_appliquee", "acceptee_mais_non_appliquee"])

Import fact_batches_calculation_date_reco

In [0]:
current_process="fact_batches_calculation_date_reco"

In [0]:
target_fact_batches_calculation_date_reco = current_catalog +"."+current_schema+"."+current_process
print(target_fact_batches_calculation_date_reco)

In [0]:
all_columns =  df_batches_calculation_date_reco.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'batch'
    ,'calculation_date']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_batches_calculation_date_reco, 
    target_fact_batches_calculation_date_reco, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

dim_batch

In [0]:
dim_batch = f"""
SELECT id_batch, batch_number, mes_number, id_plant, id_cdc, id_variety, harvest, batch_status, planned_date
FROM batches
"""

df_dim_batch = spark.sql(dim_batch)

# POLISY1
# df_dim_batch = df_dim_batch.filter((F.col('id_plant') != 2) | (F.col('planned_date') > F.lit('2024-02-13')))

# # NOGENT2
# df_dim_batch = df_dim_batch.filter((F.col('id_plant') != 3) | (F.col('planned_date') > F.lit('2024-05-13')))

# # PROUVY1
# df_dim_batch = df_dim_batch.filter((F.col('id_plant') != 4) | (F.col('planned_date') > F.lit('2024-06-09')))

# # STRASBOURG2
# df_dim_batch = df_dim_batch.filter((F.col('id_plant') != 5) | (F.col('planned_date') > F.lit('2024-08-18')))

# # NOGENT1
# df_dim_batch = df_dim_batch.filter((F.col('id_plant') != 6) | (F.col('planned_date') > F.lit('2024-09-16')))

Import dim_batch

In [0]:
current_process="dim_batch"

In [0]:
target_dim_batch = current_catalog +"."+current_schema+"."+current_process
print(target_dim_batch)

In [0]:
all_columns =  df_dim_batch.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_batch']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_dim_batch, 
    target_dim_batch, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

dim_specification

In [0]:
dim_cdc = f"""
SELECT id_requirement_specification AS id_cdc, name AS cdc
FROM {source}.requirement_specifications
WHERE deleted = false
"""

df_dim_cdc = spark.sql(dim_cdc)

Import dim_specification

In [0]:
current_process="dim_specification"

In [0]:
target_dim_specification = current_catalog +"."+current_schema+"."+current_process
print(target_dim_specification)

In [0]:
all_columns =  df_dim_cdc.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_cdc']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_dim_cdc, 
    target_dim_specification, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

dim_localization

In [0]:
dim_localization = f"""
WITH
localizations AS (
    SELECT
        pl.id_parameter_localization AS id_localization,
        MAX(CASE WHEN plt.language = 1 THEN plt.label END) AS localization,
        MAX(CASE WHEN plt.language = 2 THEN plt.label END) AS localization_eng,
        pl.sequence
    FROM {source}.parameters_localizations pl
    JOIN {source}.parameters_localizations_translations plt
        ON plt.id_parameter_localization = pl.id_parameter_localization
        AND plt.deleted = false
    WHERE pl.deleted = false
    GROUP BY pl.id_parameter_localization, pl.sequence
)
SELECT * FROM localizations
"""

df_dim_localization = spark.sql(dim_localization)

Import dim_localization

In [0]:
current_process="dim_localization"

In [0]:
target_dim_localization = current_catalog +"."+current_schema+"."+current_process
print(target_dim_localization)

In [0]:
all_columns =  df_dim_localization.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_localization']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_dim_localization, 
    target_dim_localization, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

dim_profiles_users

In [0]:
dim_profiles_users = f"""
SELECT id_profile_user, name, surname
FROM {source}.profiles_users
WHERE deleted = false
"""

df_dim_profiles_users = spark.sql(dim_profiles_users)

Import dim_profiles_users

In [0]:
current_process="dim_profiles_users"

In [0]:
target_dim_profiles_users = current_catalog +"."+current_schema+"."+current_process
print(target_dim_profiles_users)

In [0]:
all_columns =  df_dim_profiles_users.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_profile_user']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_dim_profiles_users, 
    target_dim_profiles_users, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

dim_site

In [0]:
dim_site = f"""
SELECT id_plant, plant
FROM batches
GROUP BY id_plant, plant
ORDER BY id_plant ASC
"""

df_dim_site = spark.sql(dim_site)

Import dim_site

In [0]:
current_process="dim_site"

In [0]:
target_dim_site = current_catalog +"."+current_schema+"."+current_process
print(target_dim_site)

In [0]:
all_columns =  df_dim_site.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_plant']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_dim_site, 
    target_dim_site, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

dim_variety_specy

In [0]:
dim_variety_specy = f"""
SELECT gv.id_good_variety AS id_variety, gv.code AS variety, gs.code AS specy
FROM {source}.goods_varieties gv
JOIN {source}.goods_species gs ON gs.id_good_specy = gv.specy AND gs.deleted = false
WHERE gv.deleted = false
ORDER BY id_variety
"""

df_dim_variety_specy = spark.sql(dim_variety_specy)

Import dim_variety_specy

In [0]:
current_process="dim_variety_specy"

In [0]:
target_dim_variety_specy = current_catalog +"."+current_schema+"."+current_process
print(target_dim_variety_specy)

In [0]:
all_columns =  df_dim_variety_specy.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_variety']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_dim_variety_specy, 
    target_dim_variety_specy, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )